# 15 - Does a guide with reference values help the reader?

The facts alone did not help (notebook 14): they were raw numbers, and the reader never saw what laundering looks like in this bank. Two guided conditions on the development sample, against K1:

- **K1_guide**: each fact beside the reference accounts (share with a lower and with the same value, and the 10th-90th percentile of equally busy clean accounts), and a short guide (`guide.GUIDE`).
- **K1_guide_shots**: the same with four solved examples (`guide.SHOT_RULES`), one line of facts each.

Plan fixed on 14 November 2026, before any guided answer was read: measures and rule as in notebook 14 (seed 15); reference accounts = all accounts minus the two reading samples (514,568); the condition with the higher mean AUC-ROC over the two models goes to the test sample (notebook 16). Both conditions also ask for a 0-100 score, a secondary measure.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import binomtest, norm
from sklearn.metrics import roc_auc_score

from skeptical_second_reader import guide
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.pipeline import READ_ALONE, READ_GUIDE, READ_SHOTS
from skeptical_second_reader.reading_sample import (
    MATCHED_NAME,
    OUTSIDE_NAME,
    RANDOM_NAME,
)
from skeptical_second_reader.second_reader import NUM_CTX, SUSPICIOUS

MODELS = ["gemma4:latest", "qwen3.5:9b-q4_K_M"]
GUIDED = [READ_GUIDE, READ_SHOTS]
CONDITIONS = [READ_ALONE, *GUIDED]
CLEAN = [MATCHED_NAME, RANDOM_NAME]
DRAWS = 2000
SEED = 15

reading_dir = DATA_INTERIM / "reading"
dev = pd.read_parquet(DATA_INTERIM / "reading_dev_cases.parquet")["stratum"]
test = pd.read_parquet(DATA_INTERIM / "reading_test_cases.parquet")["stratum"]
alone = pd.read_parquet(reading_dir / "results_dev.parquet")
guided = pd.read_parquet(reading_dir / "results_dev_guided.parquet")
prompts = pd.read_parquet(reading_dir / "prompts_dev_guided.parquet")
results = pd.concat(
    [alone[alone["condition"] == READ_ALONE], guided], ignore_index=True
)
ceilings = pd.read_parquet(DATA_INTERIM / "ceiling_facts_scores.parquet")
truth = (~dev.isin(CLEAN)).astype(int)
patterns = sorted(s for s in dev.unique() if s.startswith("pattern "))
ORDER = patterns + [OUTSIDE_NAME] + CLEAN
print(f"accounts {len(dev)}  positives {truth.sum()}  clean {(truth == 0).sum()}")

accounts 255  positives 135  clean 120


## 1. The guide and the examples, checked against the reference accounts

Every rate the guide states is recomputed from the reference accounts, and each example is checked against its rule and against the reading samples. The guide is printed as the reader sees it.

In [2]:
facts = pd.read_parquet(DATA_INTERIM / "account_facts.parquet")
labels = pd.read_parquet(DATA_INTERIM / "transaction_reading.parquet")["label"]
activity = pd.read_parquet(DATA_INTERIM / "ceiling_features.parquet")["transactions"]
used = pd.read_parquet(DATA_INTERIM / "llm_experiment_cases.parquet").index.union(
    pd.read_parquet(DATA_INTERIM / "llm_trial_cases.parquet").index
)
reference = facts.index.difference(dev.index).difference(test.index)
f, y_ref, act = facts.loc[reference], labels.loc[reference], activity.loc[reference]
conv = f["sends_with_conversion"]
busy = act >= 10


def rate(mask):
    return f"{y_ref[mask].mean() * 100:.1f}% of {int(mask.sum()):,}"


checks = {
    "all reference accounts": rate(pd.Series(True, index=reference)),
    "10+ transactions": rate(busy),
    "10+ tx, same amount 5+ times": rate(busy & (f["most_sends_same_amount"] >= 5)),
    "10+ tx, same amount under 5": rate(busy & (f["most_sends_same_amount"] < 5)),
    "10+ tx, sent on within a day 80%+": rate(
        busy & (f["share_sent_on_within_day"] >= 0.8)
    ),
    "10+ tx, sent on within a day under 80%": rate(
        busy & (f["share_sent_on_within_day"] < 0.8)
    ),
    "exactly 1 conversion-linked payment": rate(conv == 1),
    "exactly 1, at most 1 payee": rate((conv == 1) & (f["payees"] <= 1)),
    "exactly 2": rate(conv == 2),
    "3 or more": rate(conv >= 3),
    "a counterparty both ways": rate(f["both_ways"] >= 1),
    "9 or more payers": rate(f["payers"] >= 9),
}
print(f"reference accounts {len(reference):,}")
for name, text in checks.items():
    print(f"  {name}: {text} involved")

shots = guide.choose_shots(
    facts, labels, activity, dev.index.union(test.index).union(used), seed=SEED
)
masks = guide.shot_candidates(
    facts.loc[shots.index], labels.loc[shots.index], activity.loc[shots.index]
)
for account, rule in shots.items():
    assert masks[rule][account], rule
assert not shots.index.isin(dev.index.union(test.index).union(used)).any()
print()
print("examples:", shots.to_dict())
print()
print(guide.GUIDE)
print()
print(guide.shots_text(facts, shots, activity))

reference accounts 514,568
  all reference accounts: 1.2% of 514,568 involved
  10+ transactions: 1.8% of 216,829 involved
  10+ tx, same amount 5+ times: 1.9% of 102,450 involved
  10+ tx, same amount under 5: 1.8% of 114,379 involved
  10+ tx, sent on within a day 80%+: 1.8% of 67,449 involved
  10+ tx, sent on within a day under 80%: 1.9% of 139,096 involved
  exactly 1 conversion-linked payment: 35.6% of 2,534 involved
  exactly 1, at most 1 payee: 68.5% of 495 involved
  exactly 2: 8.7% of 1,436 involved
  3 or more: 3.1% of 4,483 involved
  a counterparty both ways: 9.1% of 6,667 involved
  9 or more payers: 8.9% of 3,375 involved



examples: {'02591:80211CB70': 'strong sign', '0212789:804FAD580': 'weaker sign', '019474:803AE9360': 'busy clean', '021611:8016DEDA0': 'many conversions'}

GUIDE: WHAT SEPARATES LAUNDERING ACCOUNTS IN THIS BANK
These rates were measured on other accounts of this bank, never on the account below.
- About 1 account in 85 (1.2%) is involved in money laundering. Most accounts are clean, so answer suspicious only when you see one of the signs below.
- Not signs in this bank, because clean accounts show them just as often (among accounts with 10 or more transactions, about 2% are involved with or without them): many transactions, sending the same amount many times, sending money on within 24 hours of receiving it, large amounts, cash, cheques or credit cards.
- Strongest sign: a payment to another account in the same minute as a currency conversion by A (in the table, a "self" row whose paid and received currencies differ and a "sent" row at the same time). Of accounts with exactly one such

## 2. Integrity

Checks that every account was read once per condition with one model version, how many answers are not usable, and that no prompt was cut by the 4,096-token window.

In [3]:
assert set(results["model"]) == set(MODELS)
assert set(results["condition"]) == set(CONDITIONS)
assert not results.duplicated(["model", "condition", "account"]).any()
for (model, condition), part in results.groupby(["model", "condition"]):
    assert set(part["account"]) == set(dev.index), (model, condition)
assert (results["group"] == dev.loc[results["account"]].to_numpy()).all()
assert (results.groupby("model")["digest"].nunique() == 1).all()
assert results["think"].nunique() == 1
assert len(prompts) == 2 * len(dev) and set(prompts["condition"]) == set(GUIDED)
assert prompts["prompt"].str.contains("GUIDE: WHAT SEPARATES").all()
assert (
    prompts.loc[prompts["condition"] == READ_SHOTS, "prompt"]
    .str.contains("SOLVED EXAMPLES")
    .all()
)
assert (
    not prompts.loc[prompts["condition"] == READ_GUIDE, "prompt"]
    .str.contains("SOLVED EXAMPLES")
    .any()
)

results["usable"] = results["decision"].notna() & results["confidence"].notna()
results["total_tokens"] = results["prompt_tokens"] + results["answer_tokens"]
tokens = results.groupby(["model", "condition"]).agg(
    answers=("account", "size"),
    not_usable=("usable", lambda s: int((~s).sum())),
    score_missing=("score", lambda s: int(s.isna().sum())),
    max_prompt_tokens=("prompt_tokens", "max"),
    max_prompt_and_answer=("total_tokens", "max"),
    at_window=("prompt_tokens", lambda s: int((s >= NUM_CTX).sum())),
)
print(f"context window {NUM_CTX}")
print(tokens.to_string())

context window 4096
                                  answers  not_usable  score_missing  max_prompt_tokens  max_prompt_and_answer  at_window
model             condition                                                                                              
gemma4:latest     K1                  255           0            255               1546                   1784          0
                  K1_guide            255           0              0               2613                   2849          0
                  K1_guide_shots      255           0              0               3024                   3175          0
qwen3.5:9b-q4_K_M K1                  255           0            255               1570                   1743          0
                  K1_guide            255           0              0               2641                   2880          0
                  K1_guide_shots      255           0              0               3053                   3178          0


## 3. Share flagged per stratum

Share of accounts called suspicious, with a Wilson 95% interval: recall for the eight pattern types and the positives outside the file, false alarm rate for the two clean strata (60 accounts each; the others 15).

In [4]:
def wilson(k, n, level=0.95):
    if n == 0:
        return np.nan, np.nan
    z = norm.ppf(0.5 + level / 2)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


usable_all = results.groupby(["model", "account"])["usable"].all()
paired = {m: usable_all[m][usable_all[m]].index for m in MODELS}
use = results.copy()
use = use[[a in paired[m] for m, a in zip(use["model"], use["account"], strict=True)]]
use["says"] = use["decision"] == SUSPICIOUS
use["conf_score"] = np.where(use["says"], use["confidence"], 1 - use["confidence"])
for m in MODELS:
    print(f"{m}: accounts usable in every condition {len(paired[m])}")

rows = []
for (model, condition, stratum), part in use.groupby(["model", "condition", "group"]):
    k, n = int(part["says"].sum()), len(part)
    low, high = wilson(k, n)
    rows.append([model, condition, stratum, k, n, k / n, low, high])
flagged = pd.DataFrame(
    rows,
    columns=["model", "condition", "stratum", "flagged", "n", "share", "low", "high"],
)
for model in MODELS:
    mine = flagged[flagged["model"] == model].copy()
    mine["text"] = mine.apply(
        lambda r: (
            f"{r['flagged']:>2}/{r['n']:<2} {r['share']:.2f} "
            f"[{r['low']:.2f}, {r['high']:.2f}]"
        ),
        axis=1,
    )
    print(model)
    print(
        mine.pivot(index="stratum", columns="condition", values="text")
        .reindex(ORDER)[CONDITIONS]
        .to_string()
    )
    print()

gemma4:latest: accounts usable in every condition 255
qwen3.5:9b-q4_K_M: accounts usable in every condition 255
gemma4:latest
condition                                        K1                 K1_guide           K1_guide_shots
stratum                                                                                              
pattern BIPARTITE            9/15 0.60 [0.36, 0.80]   1/15 0.07 [0.01, 0.30]   3/15 0.20 [0.07, 0.45]
pattern CYCLE               11/15 0.73 [0.48, 0.89]   3/15 0.20 [0.07, 0.45]   4/15 0.27 [0.11, 0.52]
pattern FAN-IN              15/15 1.00 [0.80, 1.00]   6/15 0.40 [0.20, 0.64]   9/15 0.60 [0.36, 0.80]
pattern FAN-OUT              4/15 0.27 [0.11, 0.52]   2/15 0.13 [0.04, 0.38]   2/15 0.13 [0.04, 0.38]
pattern GATHER-SCATTER       6/15 0.40 [0.20, 0.64]   2/15 0.13 [0.04, 0.38]   3/15 0.20 [0.07, 0.45]
pattern RANDOM               7/15 0.47 [0.25, 0.70]   3/15 0.20 [0.07, 0.45]   2/15 0.13 [0.04, 0.38]
pattern SCATTER-GATHER      11/15 0.73 [0.48, 0.89]   8/15

## 4. Paired comparison against K1

The main result: for each model and guided condition, the AUC-ROC change against K1 on the same accounts with a 95% interval, and how many decisions became right or wrong (McNemar, positives and clean apart).

In [5]:
def draws_for(accounts, seed):
    rng = np.random.default_rng(seed)
    strata = dev.loc[accounts].to_numpy()
    index = [np.flatnonzero(strata == s) for s in ORDER]
    return [
        np.concatenate(
            [rng.choice(ix, size=len(ix), replace=True) for ix in index if len(ix)]
        )
        for _ in range(DRAWS)
    ]


def paired_change(y, base, new, draws):
    a, b = roc_auc_score(y, base), roc_auc_score(y, new)
    boot = np.array(
        [roc_auc_score(y[d], new[d]) - roc_auc_score(y[d], base[d]) for d in draws]
    )
    low, high = np.percentile(boot, [2.5, 97.5])
    rule = "helped" if low > 0 else ("hurt" if high < 0 else "a help was not shown")
    return a, b, b - a, low, high, rule


mcnemar_rows, auc_rows, wides = [], [], {}
for model in MODELS:
    accounts = paired[model]
    y = truth.loc[accounts].to_numpy()
    draws = draws_for(accounts, SEED)
    wide = (
        use[use["model"] == model]
        .pivot(
            index="account", columns="condition", values=["says", "conf_score", "score"]
        )
        .loc[accounts]
    )
    wides[model] = (wide, y, draws)
    right = {
        c: wide[("says", c)].astype(bool).to_numpy() == (y == 1) for c in CONDITIONS
    }
    for condition in GUIDED:
        for side, mask in [("positives", y == 1), ("clean", y == 0)]:
            better = int((~right[READ_ALONE] & right[condition] & mask).sum())
            worse = int((right[READ_ALONE] & ~right[condition] & mask).sum())
            p = (
                binomtest(better, better + worse, 0.5).pvalue
                if better + worse
                else np.nan
            )
            mcnemar_rows.append(
                [
                    model,
                    condition,
                    side,
                    int(mask.sum()),
                    int((right[READ_ALONE] & mask).sum()),
                    int((right[condition] & mask).sum()),
                    better,
                    worse,
                    p,
                ]
            )
        base = wide[("conf_score", READ_ALONE)].to_numpy(float)
        new = wide[("conf_score", condition)].to_numpy(float)
        auc_rows.append([model, condition, *paired_change(y, base, new, draws)])

mcnemar = pd.DataFrame(
    mcnemar_rows,
    columns=[
        "model",
        "condition",
        "accounts",
        "n",
        "right K1",
        "right guided",
        "became right",
        "became wrong",
        "exact p",
    ],
)
auc = pd.DataFrame(
    auc_rows,
    columns=[
        "model",
        "condition",
        "AUC K1",
        "AUC guided",
        "change",
        "low 95%",
        "high 95%",
        "rule",
    ],
)
print("McNemar, decisions that change against K1")
print(mcnemar.round(4).to_string(index=False))
print()
print(f"AUC-ROC (confidence score), stratified paired bootstrap {DRAWS}")
print(auc.round(3).to_string(index=False))

McNemar, decisions that change against K1
            model      condition  accounts   n  right K1  right guided  became right  became wrong  exact p
    gemma4:latest       K1_guide positives 135        81            30             2            53      0.0
    gemma4:latest       K1_guide     clean 120        68           114            47             1      0.0
    gemma4:latest K1_guide_shots positives 135        81            32             6            55      0.0
    gemma4:latest K1_guide_shots     clean 120        68           115            48             1      0.0
qwen3.5:9b-q4_K_M       K1_guide positives 135        99            48             3            54      0.0
qwen3.5:9b-q4_K_M       K1_guide     clean 120        45           117            72             0      0.0
qwen3.5:9b-q4_K_M K1_guide_shots positives 135        99            47             3            55      0.0
qwen3.5:9b-q4_K_M K1_guide_shots     clean 120        45           117            72          

## 5. Secondary measures

AUC of the decision alone (balanced accuracy), AUC of the 0-100 score, and the share of tied scores between positive and clean accounts: whether the gain is in the decisions or only in the ranking.

In [6]:
def tie_share(y, s):
    pos, neg = s[y == 1], s[y == 0]
    return float((pos[:, None] == neg[None, :]).mean())


rows = []
for model in MODELS:
    wide, y, draws = wides[model]
    for condition in CONDITIONS:
        says = wide[("says", condition)].astype(float).to_numpy()
        conf = wide[("conf_score", condition)].to_numpy(float)
        row = {
            "model": model,
            "condition": condition,
            "AUC confidence": roc_auc_score(y, conf),
            "AUC decision only": roc_auc_score(y, says),
            "distinct confidence values": len(np.unique(conf)),
            "tied pairs, confidence": tie_share(y, conf),
        }
        if condition in GUIDED:
            score = wide[("score", condition)].to_numpy(float)
            ok = ~np.isnan(score)
            row["score answers"] = int(ok.sum())
            row["AUC 0-100 score"] = (
                roc_auc_score(y[ok], score[ok])
                if ok.all() or len(np.unique(y[ok])) == 2
                else np.nan
            )
            row["tied pairs, 0-100 score"] = tie_share(y[ok], score[ok])
        rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))

print()
print("0-100 score change against the K1 confidence score (descriptive, same draws)")
for model in MODELS:
    wide, y, draws = wides[model]
    base = wide[("conf_score", READ_ALONE)].to_numpy(float)
    for condition in GUIDED:
        score = wide[("score", condition)].to_numpy(float)
        if np.isnan(score).any():
            print(
                f"  {model} {condition}: {int(np.isnan(score).sum())} answers "
                "without a score, skipped"
            )
            continue
        a, b, d, low, high, _ = paired_change(y, base, score, draws)
        print(
            f"  {model} {condition}: {a:.3f} -> {b:.3f} "
            f"({d:+.3f} [{low:+.3f}; {high:+.3f}])"
        )

            model      condition  AUC confidence  AUC decision only  distinct confidence values  tied pairs, confidence  score answers  AUC 0-100 score  tied pairs, 0-100 score
    gemma4:latest             K1           0.597              0.583                           6                   0.216            NaN              NaN                      NaN
    gemma4:latest       K1_guide           0.673              0.586                           7                   0.303          255.0            0.672                    0.258
    gemma4:latest K1_guide_shots           0.692              0.598                           7                   0.330          255.0            0.695                    0.336
qwen3.5:9b-q4_K_M             K1           0.536              0.554                           5                   0.418            NaN              NaN                      NaN
qwen3.5:9b-q4_K_M       K1_guide           0.690              0.665                           6                   0

  gemma4:latest K1_guide: 0.597 -> 0.672 (+0.075 [+0.021; +0.131])


  gemma4:latest K1_guide_shots: 0.597 -> 0.695 (+0.098 [+0.039; +0.161])


  qwen3.5:9b-q4_K_M K1_guide: 0.536 -> 0.710 (+0.174 [+0.107; +0.245])


  qwen3.5:9b-q4_K_M K1_guide_shots: 0.536 -> 0.712 (+0.176 [+0.110; +0.244])


## 6. Beside the ceiling

The classifier trained on the prompt evidence and facts, on the same accounts: what the same information gives a trained model.

In [7]:
side = {}
accounts_all = dev.index
for name in ["prompt evidence", "evidence and facts"]:
    side[f"ceiling {name}"] = (
        (ceilings.loc[accounts_all, f"flags {name}"] == 1).groupby(dev).mean()
    )
for model in MODELS:
    for condition in CONDITIONS:
        part = use[(use["model"] == model) & (use["condition"] == condition)]
        side[f"{model.split(':')[0]} {condition}"] = part.groupby("group")[
            "says"
        ].mean()
print("share flagged per stratum")
print(pd.DataFrame(side).reindex(ORDER).round(2).to_string())
print()
y_all = truth.loc[accounts_all].to_numpy()
aucs = {
    f"ceiling {n}": roc_auc_score(y_all, ceilings.loc[accounts_all, f"scores {n}"])
    for n in ["prompt evidence", "evidence and facts"]
}
for row in auc_rows:
    aucs[f"{row[0].split(':')[0]} {row[1]}"] = row[3]
for model in MODELS:
    aucs[f"{model.split(':')[0]} {READ_ALONE}"] = auc.loc[
        auc["model"] == model, "AUC K1"
    ].iloc[0]
print("AUC-ROC (readers: accounts usable for that model)")
print(pd.Series(aucs).round(3).to_string())

share flagged per stratum
                            ceiling prompt evidence  ceiling evidence and facts  gemma4 K1  gemma4 K1_guide  gemma4 K1_guide_shots  qwen3.5 K1  qwen3.5 K1_guide  qwen3.5 K1_guide_shots
pattern BIPARTITE                              0.27                        0.27       0.60             0.07                   0.20        0.73              0.27                    0.27
pattern CYCLE                                  0.60                        0.53       0.73             0.20                   0.27        0.93              0.53                    0.53
pattern FAN-IN                                 0.80                        0.80       1.00             0.40                   0.60        0.93              0.73                    0.73
pattern FAN-OUT                                0.20                        0.40       0.27             0.13                   0.13        0.33              0.20                    0.13
pattern GATHER-SCATTER                         0.

## 7. Condition for the test sample

The guided condition with the higher mean AUC-ROC over the two models; the test sample is read once in it (notebook 16).

In [8]:
mean_auc = auc.groupby("condition")["AUC guided"].mean()
chosen = mean_auc.idxmax()
print(mean_auc.round(3).to_string())
print(f"condition for the test sample: {chosen}")

condition
K1_guide          0.681
K1_guide_shots    0.688
condition for the test sample: K1_guide_shots


## 8. Results

- Both guided conditions helped both models. AUC-ROC: Gemma 0.597 to 0.673 (K1_guide) and 0.692 (K1_guide_shots); Qwen 0.536 to 0.690 and 0.685. Every interval is above 0.
- The decisions moved far towards `not suspicious`: false alarms fell from 30/60 and 22/60 (Gemma) and 41/60 and 34/60 (Qwen) to at most 4/60 per clean stratum, and recall fell in every positive stratum. Of the positives, 53 to 55 became wrong and 2 to 6 right.
- The ranking improved more than the decisions: decision-only AUC rose by at most 0.015 (Gemma) and 0.11 (Qwen), confidence AUC by 0.075 to 0.154.
- Positives outside the pattern file are almost never flagged (0 to 3 of 15): the guide names no sign for them.
- The examples added little (Gemma +0.019, Qwen -0.005 against K1_guide); K1_guide_shots goes to the test sample (mean 0.688 against 0.681).
- The guide's strongest sign (one payment in the same minute as a conversion, notebook 06) is probably a mark of the simulator, so part of the gain may not carry over to other data.